In [ ]:
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import date

ENDPOINT = "http://www.semanticweb.org/nicol/ontologies/2025/9/traffic"

PREFIXES = """
PREFIX :   <http://www.semanticweb.org/nicol/ontologies/2025/9/traffic#>
PREFIX geo: <http://www.opengis.net/ont/geosparql#>
PREFIX geof: <http://www.opengis.net/def/function/geosparql#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX sf: <http://www.opengis.net/ont/sf#>
PREFIX traffic: <http://www.semanticweb.org/nicol/ontologies/2025/9/traffic>
PREFIX xml:  <http://www.w3.org/XML/1998/namespace>
PREFIX xsd:  <http://www.w3.org/2001/XMLSchema#>
"""

def run_sparql(query: str) -> pd.DataFrame:
    """Run a SELECT query and return a pandas DataFrame."""
    sparql = SPARQLWrapper(ENDPOINT)
    sparql.setReturnFormat(JSON)
    sparql.setQuery(PREFIXES + query)
    results = sparql.query().convert()
    cols = [b['name'] for b in results['head']['vars']]
    rows = []
    for b in results['results']['bindings']:
        row = []
        for c in cols:
            row.append(b[c]['value'] if c in b else None)
        rows.append(row)
    df = pd.DataFrame(rows, columns=cols)
    return df

In [ ]:
def q1_flow_accidents(segment_iri: str, start="2025-01-01", end="2025-10-01", bucket="month"):
    # time frame can be selected (bucket) either "day" or "month"
    if bucket == "day":
        bucket_expr = 'CONCAT(STR(YEAR(?t)),"-",SUBSTR(CONCAT("0",STR(MONTH(?t))),STRLEN(STR(MONTH(?t)))),"-",SUBSTR(CONCAT("0",STR(DAY(?t))),STRLEN(STR(DAY(?t)))))'
        acc_bucket_expr = 'CONCAT(STR(YEAR(?ta)),"-",SUBSTR(CONCAT("0",STR(MONTH(?ta))),STRLEN(STR(MONTH(?ta)))),"-",SUBSTR(CONCAT("0",STR(DAY(?ta))),STRLEN(STR(DAY(?ta)))))'
    else:  # month
        bucket_expr = 'CONCAT(STR(YEAR(?t)),"-",SUBSTR(CONCAT("0",STR(MONTH(?t))),STRLEN(STR(MONTH(?t)))))'
        acc_bucket_expr = 'CONCAT(STR(YEAR(?ta)),"-",SUBSTR(CONCAT("0",STR(MONTH(?ta))),STRLEN(STR(MONTH(?ta)))))'

    query = f"""
    SELECT ?period (AVG(?flow) AS ?avgFlow) (AVG(?speed) AS ?avgSpeed) (COUNT(?acc) AS ?accidents)
    WHERE {{
      VALUES ?segment {{ <{segment_iri}> }}
      VALUES (?start ?end) {{ ("{start}"^^xsd:date "{end}"^^xsd:date) }}

      ?obs a ex:TrafficObservation ;
           ex:observedOn ?segment ;
           ex:windowStart ?t ;
           ex:avgFlow ?flow ;
           ex:avgSpeed ?speed .
      FILTER (?t >= ?start && ?t < ?end)

      OPTIONAL {{
        ?acc a ex:Accident ;
             ex:happenedOn ?segment ;
             ex:accidentTime ?ta .
        FILTER (?ta >= ?start && ?ta < ?end)
        BIND({acc_bucket_expr} AS ?accMonth)
      }}

      BIND({bucket_expr} AS ?period)
    }}
    GROUP BY ?period
    ORDER BY ?period
    """
    df = run_sparql(query)
    for c in ["avgFlow", "avgSpeed", "accidents"]:
        if c in df:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def q1_correlation_plot(df):
    # using a pearson to represent correlation
    corr = df[["avgFlow","accidents"]].dropna().corr(method="pearson").iloc[0,1]
    print(f"Pearson r (avgFlow vs accidents): {corr:.3f}")
    sns.regplot(data=df, x="avgFlow", y="accidents", ci=None)
    plt.title(f"Flow vs Accidents (r={corr:.2f})")
    plt.show()


In [ ]:
def q2_severity_distribution():
    query = """
    SELECT ?infraType ?severity (COUNT(*) AS ?n)
    WHERE {
      ?a a ex:Accident ;
         ex:nearInfrastructure ?i ;
         ex:severity ?severity .
      ?i a ex:Infrastructure ;
         ex:infrastructureType ?infraType .
    }
    GROUP BY ?infraType ?severity
    ORDER BY ?infraType ?severity
    """
    df = run_sparql(query)
    if "n" in df: df["n"] = pd.to_numeric(df["n"], errors="coerce")
    return df.pivot_table(index="infraType", columns="severity", values="n", fill_value=0)


In [ ]:
def q3_vehicle_types_peak_hours():
    query = """
    SELECT ?vehicleCategory (COUNT(DISTINCT ?a) AS ?accidents)
    WHERE {
      ?a a ex:Accident ;
         ex:inArea ?area ;
         ex:accidentTime ?t ;
         ex:involvesVehicle ?veh .
      ?area a ?areaType .
      VALUES ?areaType { ex:ResidentialArea ex:SchoolZone }
      ?veh a ex:Vehicle ; ex:vehicleCategory ?vehicleCategory .
      BIND(HOUR(?t) AS ?h)
      FILTER ((?h >= 7 && ?h < 9) || (?h >= 16 && ?h < 18))
    }
    GROUP BY ?vehicleCategory
    ORDER BY DESC(?accidents)
    """
    df = run_sparql(query)
    if "accidents" in df: df["accidents"] = pd.to_numeric(df["accidents"], errors="coerce")
    return df


In [ ]:
def q4_segments_ped_cyc_weather(limit=20):
    query = f"""
    SELECT ?segment (COUNT(DISTINCT ?a) AS ?accidents)
    WHERE {{
      ?segment a ex:RoadSegment ; ex:speedLimit ?limit .
      FILTER (?limit > 30)
      ?a a ex:Accident ;
         ex:happenedOn ?segment ;
         ex:weatherCondition ?w ;
         ex:involvesActor ?actor .
      VALUES ?w {{ ex:Rainy ex:Foggy }}
      ?actor a ?actorType .
      VALUES ?actorType {{ ex:Pedestrian ex:Cyclist }}
    }}
    GROUP BY ?segment
    ORDER BY DESC(?accidents)
    LIMIT {int(limit)}
    """
    df = run_sparql(query)
    if "accidents" in df: df["accidents"] = pd.to_numeric(df["accidents"], errors="coerce")
    return df


In [ ]:
def q5_persistent_hotspots(years=(2021,2022,2023,2024,2025), threshold=10, min_years=3):
    years_values = " ".join(str(y) for y in years)
    query = f"""
    SELECT ?zone (COUNT(*) AS ?yearsAboveThreshold) (SUM(?cnt) AS ?totalAccidents)
    WHERE {{
      {{
        SELECT ?zone ?yr (COUNT(?a) AS ?cnt)
        WHERE {{
          VALUES ?yr {{ {years_values} }}
          ?a a ex:Accident ; ex:inZone ?zone ; ex:accidentTime ?t .
          FILTER (YEAR(?t) = ?yr)
        }}
        GROUP BY ?zone ?yr
      }}
      FILTER (?cnt >= {threshold})
    }}
    GROUP BY ?zone
    HAVING (COUNT(*) >= {min_years})
    ORDER BY DESC(?yearsAboveThreshold) DESC(?totalAccidents)
    """
    df = run_sparql(query)
    for c in ["yearsAboveThreshold","totalAccidents"]:
        if c in df: df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


In [ ]:
def q6_city_frequencies_nl(start="2024-01-01", end="2025-01-01"):
    query = f"""
    SELECT ?city (COUNT(DISTINCT ?a) AS ?accidents)
    WHERE {{
      ?a a ex:Accident ; ex:inCity ?city ; ex:accidentTime ?t .
      ?city ex:inCountry ex:Netherlands .
      FILTER (?t >= "{start}"^^xsd:date && ?t < "{end}"^^xsd:date)
    }}
    GROUP BY ?city
    ORDER BY DESC(?accidents)
    """
    df = run_sparql(query)
    if "accidents" in df: df["accidents"] = pd.to_numeric(df["accidents"], errors="coerce")
    return df


In [ ]:
if __name__ == "__main__":
    # Q1: correlation data plot
    seg = "http://example.org/traffic#RoadSegment_123"
    df_q1 = q1_flow_accidents(seg, start="2025-01-01", end="2025-10-01", bucket="month")
    print(df_q1.head())
    q1_correlation_plot(df_q1)

    # Q2: accident severity and infrastructure type graph
    print(q2_severity_distribution())

    # Q3: peak-hours vehicles
    print(q3_vehicle_types_peak_hours())

    # Q4: hazardous (>30 km/h) segments for ped/cyclist in rain/fog
    print(q4_segments_ped_cyc_weather(limit=15))

    # Q5: persistent hotspots
    print(q5_persistent_hotspots(years=(2021,2022,2023,2024,2025), threshold=12, min_years=3))

    # Q6: per-city counts for 2024
    print(q6_city_frequencies_nl())
